In [0]:

# Project         : Procurement Analytics using Databricks & Power BI
# Layer           : Bronze
# Notebook        : bronze_goods_receipts
# Source          : payments.csv
# Target          : procurement.bronze.bronze_goods_receipts
# Audit Table     : procurement.audit.duplicate_goods_receipts
#
# Author          : V R Mutyala
# Created Date    : 21-Jul-2026
# Last Modified   : 21-Jul-2026
#
# Description
# -----------
# This notebook loads raw goods_receipts master data into the Bronze layer.
# It validates the source data, separates duplicate records, adds audit columns, and stores the results as Delta tables.

# ==============================================================================
# Business Objective
# ==============================================================================
#
# Load Goods_Receipts master data from the source CSV file into the Bronze layer.
#
# Preserve the raw source data with minimal transformations.
#
# Detect duplicate GRN IDs and store them in the Audit schema for business review.
#
# Add audit columns to support data lineage and traceability.
#
# Create a reliable Bronze Delta table that will serve as the source for the Silver layer.
#
# ==============================================================================

In [0]:
%run ../01_Config/Config

In [0]:
%run ../05_Helper_Functions/Helper_functions

In [0]:
print(CATALOG)
print(BRONZE_GOODS_RECEIPTS)
print(AUDIT_DUPLICATE_GOODS_RECEIPTS)
print(GOODS_RECEIPTS_FILE)

In [0]:
# Import Libraries and Widgets
from pyspark.sql import DataFrame
from pyspark.sql.functions import (col, lit,current_timestamp,when,count,trim)
from pyspark.sql.types import (StructType, StructField, StringType,IntegerType,DoubleType,DecimalType)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [0]:
#Goods_Receipts schema
goods_receipts_schema = StructType([
    StructField("grn_id", StringType(), False),
    StructField("grn_date", StringType(), True),
    StructField("po_id", StringType(), True),
    StructField("quantity_ordered_ref", IntegerType(), True),
    StructField("quantity_received", IntegerType(), True),
    StructField("receipt_status", StringType(), True),
    StructField("received_by", StringType(), True)
])
# Read Goods_Receipts master data from landing volume
bronze_goods_receipts_df = (spark.read
    .format("csv")
    .option("header", True)
    .schema(goods_receipts_schema)
    .load(GOODS_RECEIPTS_FILE)
)

#Source Data validation 
print(f"Total Records : {bronze_goods_receipts_df.count()}")

print("\nSchema:")
bronze_goods_receipts_df.printSchema()

print("\nColumns:")
print(bronze_goods_receipts_df.columns)

print("\nSampledata:")
display(bronze_goods_receipts_df.limit(10))

In [0]:
# Check the NULL and Blank grn_ids
null_blank_grn_id= bronze_goods_receipts_df.filter(col("grn_id").isNull() | (trim(col("grn_id")) == ""))

print(f"Total NULL or Blank grn_ids : {null_blank_grn_id.count()}")

display(null_blank_grn_id)

In [0]:
#Check the NULL and Blank grn date
null_blank_grn_date = bronze_goods_receipts_df.filter(col("grn_date").isNull() | (trim(col("grn_date")) == ""))

print(f"Total NULL or Blank grn_date : {null_blank_grn_date.count()}")

display(null_blank_grn_date)

In [0]:
# Check the NULL and Blank po_ids
null_blank_po_id = bronze_goods_receipts_df.filter(col("po_id").isNull() | (trim(col("po_id")) == ""))

print(f"Total NULL or Blank po_ids : {null_blank_po_id.count()}")

display(null_blank_po_id)

In [0]:
#Check the negative or NULL quantity_ordered_ref
negative_quantity_ordered_ref = bronze_goods_receipts_df.filter((col("quantity_ordered_ref") < 0) | (col("quantity_ordered_ref").isNull()))

print(f"Total Null and negative amounts : {negative_quantity_ordered_ref.count()}")

display(negative_quantity_ordered_ref)

In [0]:
#Check the negative or NULL quantity_received
negative_quantity_received = bronze_goods_receipts_df.filter((col("quantity_received") < 0) | (col("quantity_received").isNull()))

print(f"Total Null and negative quantity_received : {negative_quantity_received.count()}")

display(negative_quantity_received)


In [0]:
#Check the NULL and Blank receipt_status
null_blank_receipt_status = bronze_goods_receipts_df.filter(col("receipt_status").isNull() | (trim(col("receipt_status")) == ""))

print(f"Total NULL or Blank receipt_status : {null_blank_receipt_status.count()}")

display(null_blank_receipt_status)

In [0]:
#Check the NULL and Blank received_by
null_blank_received_by = bronze_goods_receipts_df.filter(col("received_by").isNull() | (trim(col("received_by")) == ""))

print(f"Total NULL or Blank received_by : {null_blank_received_by.count()}")

display(null_blank_received_by)

In [0]:
# ============================================================
# Identify Duplicate GRN IDs igonere NULLs
# ============================================================

duplicate_grn_keys = (
    bronze_goods_receipts_df
    .filter(
        col("grn_id").isNotNull() &
        (trim(col("grn_id")) != "")
    )
    .groupBy("grn_id")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_grn_keys)

In [0]:
# ============================================================
# Identify Duplicate Goods_Receipts Records
# Business Rule: Keep the first occurrence of each Invoices ID and identify subsequent records as duplicates.
# ============================================================

window_spec = Window.partitionBy("grn_id").orderBy("grn_date")

goods_receipts_rank_df = (
    bronze_goods_receipts_df
        .join(
            duplicate_grn_keys.select("grn_id"),
            on="grn_id",
            how="inner"
        )
        .withColumn(
            "row_num",
            row_number().over(window_spec)
        )
)

display(goods_receipts_rank_df)

In [0]:
# ============================================================
# Retrieve Duplicate Goods_Receipts Records
# ============================================================

duplicate_goods_receipts = (
    goods_receipts_rank_df
        .filter(col("row_num") > 1)
        .drop("row_num")
)

print(f"Duplicate Goods_Receipts Records : {duplicate_goods_receipts.count()}")
display(duplicate_goods_receipts)


In [0]:
# ============================================================
# Add Audit Metadata for duplicate GRN IDs 
# ============================================================

duplicate_goods_receipts = (
    duplicate_goods_receipts
        .withColumn("audit_timestamp", current_timestamp())
        .withColumn("source_table", lit("Goods_Receipts"))
        .withColumn("pipeline_layer", lit("Bronze"))
        .withColumn("issue_type", lit("Duplicate Record"))
)

display(duplicate_goods_receipts)

In [0]:
# ============================================================
# Add Audit Metadata for null invoice IDs 
# ============================================================

invalid_goods_receipts = (
    null_blank_grn_id
        .withColumn("audit_timestamp", current_timestamp())
        .withColumn("source_table", lit("Goods_Receipts"))
        .withColumn("pipeline_layer", lit("Bronze"))
        .withColumn("issue_type", lit("Duplicate Record"))
)

display(invalid_goods_receipts)

In [0]:
    # ============================================================
    # Write Duplicate Records to Audit Table
    # ============================================================

    duplicate_count = duplicate_goods_receipts.count()

    if duplicate_count > 0:

        write_delta(
            df = duplicate_goods_receipts,
            table_name = AUDIT_DUPLICATE_GOODS_RECEIPTS
        )

        print(f"Successfully written {duplicate_count} duplicate record(s) to {AUDIT_DUPLICATE_GOODS_RECEIPTS}")

    else:

        print("No duplicate Goods_Receipts records found. Audit table not created.")

In [0]:
# ============================================================
# Write Invalid Invoice IDs to Audit Table
# ============================================================

invalid_count = invalid_goods_receipts.count()

if invalid_count > 0:

    write_delta(
        df = invalid_goods_receipts,
        table_name = AUDIT_INVALID_GOODS_RECEIPTS
    )

    print(f"Successfully written {invalid_count} invalid invoice record(s) to {AUDIT_INVALID_GOODS_RECEIPTS}")

else:

    print("No invalid invoice records found. Audit table not created.")

In [0]:
# ============================================================
# Add Bronze Audit Columns
# ============================================================

bronze_goods_receipts_final_df = (
    bronze_goods_receipts_df
        .withColumn("load_timestamp", current_timestamp())
        .withColumn("source_file", lit("Goods_Receipts.csv"))
)

In [0]:
# ============================================================
# Write Bronze Delta Table
# ============================================================

write_delta(
    df=bronze_goods_receipts_final_df,
    table_name=BRONZE_GOODS_RECEIPTS
)

In [0]:
# ============================================================
# Validate Bronze Delta Table
# ============================================================

bronze_goods_receipts = spark.table(BRONZE_GOODS_RECEIPTS)

print(f"Total Bronze Records : {bronze_goods_receipts.count()}")

display(bronze_goods_receipts)

In [0]:
# ============================================================
# Bronze goods_receipts complete summary
# ============================================================
print("=" * 60)
print("Bronze Invoice Load Completed Successfully")
print("=" * 60)

print(f"{'Landing Records':<30}: {bronze_goods_receipts_df.count()}")

print(f"{'Duplicate Audit Records':<30}: {duplicate_goods_receipts.count()}")

print(f"{'Invalid Invoice Records':<30}: {invalid_goods_receipts.count()}")

print(f"{'Total Audit Records':<30}: {duplicate_goods_receipts.count() + invalid_goods_receipts.count()}")

print(f"{'Bronze Records':<30}: {spark.table(BRONZE_GOODS_RECEIPTS).count()}")